# Exercise 13.1: Exploring a comprehensive cell model

In this final exercise, we will explore how the "zoo" of currents interact to shape the mammalian action potential.

We will interface with a simplified version of the **Grandi-Bers human ventricular myocyte model**. Because this model contains dozens of state variables (ranging from membrane voltage to intracellular calcium concentrations), writing the equations out by hand would take thousands of lines of code!

Instead, the complex right-hand side (RHS) equations have been provided for you in a separate Python file called `GBV_RHS.py`. Your task is to load this file, run the simulation using `solve_ivp`, and then strategically modify the maximum conductances of specific ion channels to observe the physiological consequences.


## Exercise 13.1a: Running the baseline model

The code below sets up the wrapper required to run the external Grandi-Bers model. Run this cell to simulate a healthy human ventricular action potential.

Take note of the action potential duration (APD)—how long the voltage stays above -60 mV.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Import the external model files (ensure these are in your working directory)
import GBV_RHS as gb
import GBV_D as init


def run_grandi_bers(conductance_multipliers):
    """
    Wrapper function to run the Grandi-Bers model.
    conductance_multipliers: dictionary to scale specific currents (e.g., {'g_CaL': 1.0})
    """
    # Load default initial conditions (a large array of state variables)
    y0 = init.get_initial_conditions()

    # Load default parameters
    params = gb.get_default_parameters()

    # Apply our experimental conductance multipliers
    if "g_CaL" in conductance_multipliers:
        params["g_CaL"] *= conductance_multipliers["g_CaL"]
    if "g_Kr" in conductance_multipliers:
        params["g_Kr"] *= conductance_multipliers["g_Kr"]
    if "g_to" in conductance_multipliers:
        params["g_to"] *= conductance_multipliers["g_to"]

    # Define the time span (simulate for 600 ms to capture the whole AP)
    t_span = (0, 600)

    # Solve the system (passing the external RHS function)
    sol = solve_ivp(gb.rhs, t_span, y0, args=(params,), max_step=1.0)

    return sol.t, sol.y[0]  # sol.y[0] is the membrane voltage V_m


# Run the baseline simulation
t_base, V_base = run_grandi_bers({"g_CaL": 1.0, "g_Kr": 1.0, "g_to": 1.0})

# Plot the baseline action potential
plt.plot(t_base, V_base, "k-", linewidth=2, label="Baseline Ventricular AP")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane Potential (mV)")
plt.legend()
plt.show()

## Exercise 13.1b: Simulating Long QT Syndrome

The $I_{\mathrm{Kr}}$ current is crucial for repolarizing the cell. Many pharmaceutical drugs accidentally block the hERG channel, which reduces $I_{\mathrm{Kr}}$. This leads to a dangerous condition known as drug-induced Long QT Syndrome, where the heart takes much too long to repolarize, risking lethal arrhythmias.

Use the `run_grandi_bers` wrapper to simulate a cell where the $I_{\mathrm{Kr}}$ conductance (`g_Kr`) is reduced by 70% (i.e., multiplied by 0.3). Plot this new action potential on top of the baseline AP.


In [ ]:
# Run the modified simulation
t_lqt, V_lqt = run_grandi_bers({...})

# Plot baseline and Long QT simulation together
plt.plot(t_base, V_base, "k--", label="Baseline")
plt.plot(...)
plt.show()

## Exercise 13.1c: Ventricular vs. Atrial Cells

Atrial cells in the upper chambers of the heart have a vastly different shape than the ventricular cells you just plotted. They have a much shorter plateau phase and repolarize much faster.

This difference occurs because atrial cells express a different balance of ion channels. Specifically, atrial cells typically have:

1. A much smaller L-type calcium current ($I_{\mathrm{CaL}}$).
2. A much larger transient outward potassium current ($I_{\mathrm{to}}$).

Try to "mutate" your ventricular model into an atrial-like model. Reduce `g_CaL` by 60% (multiplier of 0.4) and increase `g_to` by 300% (multiplier of 3.0). Plot the result!


In [ ]:
# Run the atrial-like simulation
t_atrial, V_atrial = run_grandi_bers({...})

# Plot baseline and atrial simulation together
# ...